In [0]:
%run "../config/snp-cofig"

In [0]:
%run "../config/containers"

In [0]:
dbutils.widgets.text("folder","2021-03-21")

In [0]:
folder=dbutils.widgets.get("folder")

In [0]:
from pyspark.sql.types import StructType,StructField,IntegerType,StringType,DoubleType 

In [0]:
constructor_schema=StructType(
    fields=[
        StructField("constructorId",IntegerType(),True),
        StructField("constructorRef",StringType(),False),
        StructField("name",StringType(),False),
        StructField("nationality",StringType(),False),
        StructField("url",StringType(),False)
    ])

In [0]:
df=spark.read \
.schema(constructor_schema) \
.json(f"{raw_container}/{folder}/constructors.json")

In [0]:
display(df)

In [0]:
dropped_df=df.drop("url")

In [0]:
from pyspark.sql.functions import current_timestamp

final_df=dropped_df.withColumnRenamed("constructorId","constructor_id")\
    .withColumnRenamed("constructorRef","constructor_ref") \
    .withColumn("ingestion_date",current_timestamp())

In [0]:
final_df.printSchema()

In [0]:
final_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "false") \
    .partitionBy("ingestion_date")
    .saveAsTable("api_formula1.default.constructors_table")